# Sentiments — verifying (observation method) × (ground truth)

Each `ObservationMethod` and `GroundTruth` in `vonto` carries a `tags` list: a
free-form label set (`vonto.tagged.get`), always starting `["baseline", ...]`
plus the two low→high poles a `[0, 1]` value sits between (e.g.
`["baseline", "incorrect", "correct"]`).

This notebook does two things:

1. Introduces every concrete `ObservationMethod`/`GroundTruth`, and the
   *"general"* subset — usable on any single-answer trial, as opposed to
   `VarietyOM`/`ImpurityOM`/`ListVarianceGT`/`ImpurityGT`, which only make
   sense on a list-elicitation response.
2. Verifies the **sentiment construct pairing**: for each general self-report
   OM, which general computed GT is theoretically its objective counterpart —
   i.e. `sentiments = (OM × GT)` — with a small real run against Qwen.


In [1]:
import sys
import pathlib

ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "vonto").is_dir())
sys.path.insert(0, str(ROOT))

import vonto.ground_truth as GT
import vonto.observation_method as OM

print("Observation methods:")
for cls in OM.ALL:
    print(f"  {cls.__name__:14s} tags={cls.tags}")

print("\nGround truths:")
for cls in GT.ALL:
    print(f"  {cls.__name__:14s} tags={cls.tags}")


Observation methods:
  ChallengeOM    tags=['baseline', 'general', 'caves', 'defends']
  CommitmentOM   tags=['baseline', 'general', 'uncommitted', 'committed']
  WinningCommitmentOM tags=['baseline', 'uncommitted', 'committed']
  ConfidenceOM   tags=['baseline', 'general', 'unconfident', 'confident']
  ImpurityOM     tags=['baseline', 'uneven', 'even']
  NuanceOM       tags=['baseline', 'general', 'flat', 'nuanced']
  VarietyOM      tags=['baseline', 'narrow', 'varied']

Ground truths:
  ProbabilityGT  tags=['baseline', 'general', 'unlikely', 'likely']
  AnswerProbGT   tags=['baseline', 'unlikely', 'likely']
  ChallengeGT    tags=['baseline', 'general', 'caves', 'defends']
  CorrectnessGT  tags=['baseline', 'general', 'incorrect', 'correct']
  EntropyGT      tags=['baseline', 'general', 'certain', 'uncertain']
  ImpurityGT     tags=['baseline', 'uneven', 'even']
  ListVarianceGT tags=['baseline', 'narrow', 'varied']
  TemperatureGT  tags=['baseline', 'cold', 'hot']


## The general subset

`"general"` marks every OM/GT that applies to *any* single-answer trial —
everything except the list-elicitation-only pair (`VarietyOM`/`ImpurityOM`,
`ListVarianceGT`/`ImpurityGT`).

In [2]:
GENERAL_OMS = OM.get(OM.ALL)(["general"])
GENERAL_GTS = GT.get(GT.ALL)(["general"])

print("general OMs:", [c.__name__ for c in GENERAL_OMS])
print("general GTs:", [c.__name__ for c in GENERAL_GTS])


general OMs: ['ChallengeOM', 'CommitmentOM', 'ConfidenceOM', 'NuanceOM']
general GTs: ['ProbabilityGT', 'ChallengeGT', 'CorrectnessGT', 'EntropyGT']


## The sentiment pairing

Three of the four general (OM, GT) constructs line up as a self-report /
computed pair for the *same* underlying idea:

| self-report (OM)   | computed counterpart (GT) | shared idea |
|---------------------|---------------------------|-------------|
| `ConfidenceOM`       | `CorrectnessGT`           | is the answer actually right? |
| `CommitmentOM`       | `ProbabilityGT`           | how strongly did the model commit to *this exact* answer sequence when sampling it? |
| `NuanceOM`           | `EntropyGT`               | how many genuinely different answers could the model have given instead? |

`ChallengeOM`/`ChallengeGT` are excluded here — a behavioral, adversarial-
evidence protocol (needs a second, evidence-conditioned generation pass), not
a same-shape self-report/computed pair like the other three; it gets its own
row/column in `3_heatmaps.ipynb`'s Experiment 2 instead, not this notebook's
quick sanity sweep.


In [3]:
SENTIMENT_PAIRS = [
    (OM.ConfidenceOM, GT.CorrectnessGT),
    (OM.CommitmentOM, GT.ProbabilityGT),
    (OM.NuanceOM, GT.EntropyGT),
]

for om_cls, gt_cls in SENTIMENT_PAIRS:
    assert "general" in om_cls.tags and om_cls in GENERAL_OMS
    assert "general" in gt_cls.tags and gt_cls in GENERAL_GTS
    print(f"{om_cls.__name__:14s} <-> {gt_cls.__name__}")


ConfidenceOM   <-> CorrectnessGT
CommitmentOM   <-> ProbabilityGT
NuanceOM       <-> EntropyGT


## Small real verification run

A handful of real TriviaQA trials against Qwen: generate an answer, elicit
every general self-report, grade every general ground truth, and check that
each pair's own diagonal cell is at least *present* and sane (a real
correlation reading needs many more trials than fit in a quick sanity check —
that's `3_heatmaps.ipynb`'s job, not this one).

In [4]:
import shutil

from vonto import config as cfg
from vonto.models import load_model
from vonto.dataset import TriviaQA
import vonto.calibration as calib

# Wipe this demo's own calibration cache on every run -- a stale
# generation/observation/grade file has no way to tell it's stale relative to
# a code change (bitten by this more than once already: the Phase-0/Phase-1
# prompt fix, the TriviaQA alias-grading fix, and the OM category-count
# standardization, each silently masked by old cached values until manually
# cleared) -- correctness by default here is worth more than the rerun-speed
# caching bought, especially for an 8-trial sanity check that's cheap to redo
# anyway.
shutil.rmtree(pathlib.Path.cwd() / "out" / "calibration" / "sentiments_demo", ignore_errors=True)

cfg.load_credentials()
loaded = load_model("qwen")
print("model loaded:", loaded.device)


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

model loaded: cuda:0


In [5]:
N_TRIALS = 8
CACHE_DIR = pathlib.Path.cwd() / "out" / "calibration" / "sentiments_demo"

# ChallengeOM/ChallengeGT carry the "general" tag too (they apply to any
# single-answer trial, not just list-elicitation) but are excluded from this
# cross-product sweep -- see the markdown above.
SWEEP_OMS = [cls() for cls in GENERAL_OMS if cls is not OM.ChallengeOM]
SWEEP_GTS = [cls() for cls in GENERAL_GTS if cls is not GT.ChallengeGT]

triviaqa = TriviaQA(limit=30)
if triviaqa.load_if_cached((pathlib.Path.cwd() / "out" / "prepared")):
    print(f"loaded {len(triviaqa.seeds)} seeds from cache ({triviaqa.shape_tag()}.json)")
else:
    triviaqa.generate()
    triviaqa.save((pathlib.Path.cwd() / "out" / "prepared"))
print(f"{len(triviaqa.seeds)} TriviaQA seeds available, sampling {N_TRIALS} for this demo")

matrix, trials, observations, grades = calib.run_calibration(
    loaded, triviaqa, oms=SWEEP_OMS, gts=SWEEP_GTS,
    cache_dir=CACHE_DIR, n_trials=N_TRIALS, rng_seed=0,
)

om_names = [om.name for om in SWEEP_OMS]
gt_names = [gt.name for gt in SWEEP_GTS]
print("rows (OM):", om_names)
print("cols (GT):", gt_names)
print(matrix)


30 TriviaQA seeds available, sampling 8 for this demo


TriviaQA_limit30: generating trials:   0%|          | 0/8 [00:00<?, ?it/s]

TriviaQA_limit30: observation methods:   0%|          | 0/3 [00:00<?, ?it/s]

TriviaQA_limit30: commitment observations:   0%|          | 0/8 [00:00<?, ?it/s]

TriviaQA_limit30: confidence observations:   0%|          | 0/8 [00:00<?, ?it/s]

TriviaQA_limit30: nuance observations:   0%|          | 0/8 [00:00<?, ?it/s]

TriviaQA_limit30: ground truths:   0%|          | 0/3 [00:00<?, ?it/s]

rows (OM): ['commitment', 'confidence', 'nuance']
cols (GT): ['probability', 'correctness', 'entropy']
[[ 0.10910895  0.5        -0.21821789]
 [-0.05634362  0.25819889 -0.05634362]
 [-0.75592895  0.          0.75592895]]


In [6]:
print(f"{'OM':12s} {'GT':12s} {'rho':>8s}")
for om_cls, gt_cls in SENTIMENT_PAIRS:
    i = om_names.index(om_cls.name)
    j = gt_names.index(gt_cls.name)
    print(f"{om_cls.name:12s} {gt_cls.name:12s} {matrix[i, j]:8.3f}")


OM           GT                rho
confidence   correctness     0.258
commitment   probability     0.109
nuance       entropy         0.756


**Caveat:** `N_TRIALS = 8` is a structural sanity check (the pipeline runs
end to end, every cell in the matrix is a real number, nothing crashes), not
a real calibration measurement — 8 trials give a very noisy Spearman rho. The
full sweep, with enough trials per dataset to say something real about
calibration, is `3_heatmaps.ipynb`'s job.

**Historical note (no longer current):** an earlier run of this notebook saw
`commitment`/`nuance` come back constant (`nan`) — every trial landing on the
same self-reported class. That was diagnosed afterward as a symptom of
`LikertOM.observe` reading the free-form start of a fresh assistant turn
instead of a forced completion cue (fixed — see `LikertOM`'s own docstring),
not genuine model behavior; the OMs also only had 4-8 classes each at the
time, since collapsed to a shared 10-class scale across the board. Rerun this
notebook to see whether either symptom actually persists under the current
code before assuming it does.
